# Anchoring regression V5 (PFC / mFC dataset) - reproduction of El-Gaby Fig 5h

Mirror of [`LEC_elasticnet_regression_v5.ipynb`](../../code/LEC_elasticnet_regression_v5.ipynb)
on the mFC dataset. The analysis module [`elasticnet_regression_v5.py`](elasticnet_regression_v5.py)
is kept **byte-identical** to the LEC copy (`diff` it); only the loader cells below and the
grouping column differ. V4 is frozen; `elasticnet_v5_synthetics.py` control 8b asserts that V5
under the v4 flags reproduces v4 to `max|diff| = 0`.

**This dataset is El-Gaby's own.** `data/MetaData/combined_ABCDonly_days.npy` is the file
`Figure5_Regression.ipynb` loads, and the 25 recdays are `me08/me10/me11/ah03/ah04/ah07/ab03`.
So this is not "the same analysis on another region": it re-runs the published Figure 5
analysis on the published data, and each caveat below is a statement about the paper.

**What V5 changes, and why** (see `../../code/ELGABY_FIGURE5_RECONCILIATION.md`)

The deposited GitHub code was read line by line. It differs from the paper's *text* in three
places that V4 had implemented from the text:

| | paper text (V4) | his code (V5 default) |
|---|---|---|
| state-tuning statistic | peak per state and trial (`'max'`, leg-duration confounded) | **mean over the neuron's preferred-phase bins** (`'pref_phase_mean'`), NaN propagating |
| preferred phase | argmax of raw time-weighted mean rate (`'raw_mean'`) | **peak bin within each third** of the trial-averaged normalised curve (`'elgaby_peak'`) -- disagrees on 30% of PFC neuron-sessions |
| "with non-zero beta coefficients" | mean r finite | finite r in **every** fold |
| non-zero-lag neuron | majority of folds pass, all folds averaged | **>= 1 fold passes**, mean over passing folds |

Measured on the stored v4 PFC run before any re-fit: his state test gives **736** state-tuned
neurons (paper Fig 5b/c/f: 738), the finite-every-fold pool **481** (Fig 5h: 489) and **359** at
p<0.01 (ED 8b: 349); the 30-degree panel **287** vs 329; the 90-degree panel **92 vs 224** is the
residual gap this re-fit tests.

Also new: session skip reasons are recorded (`sessions_skipped`), the per-recday pooled
histogram and `mean_r_selected` use the reduced-beta r, polar rate maps per held-out task
(`*_polar.pdf`), and for Poisson runs both readouts (his linear `Xb` and the paper's
`exp(Xb+b)`) are drawn and summarised as separate rows.

**Reproduction config for this notebook**: `MIN_TRIALS = 1` (his `num_trials > 0`),
`pref_phase_source='test'` (his cell 21), ElasticNet alpha 0.01. Set `use_poisson=True` for
his *executed* estimator (~7x slower; the Poisson row then also compares to ED Fig 8d).

Gate: `python elasticnet_v5_synthetics.py` - 57 controls, including exact reproduction of v3
and v4.

In [ ]:
import numpy as np
import scipy.stats as st
from scipy import stats
from scipy.stats import zscore
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os, pickle
from tqdm import tqdm

In [ ]:
DATA_FOLDER = '/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data'
META = os.path.join(DATA_FOLDER, 'MetaData')

In [ ]:
# The canonical recday list -- the same file `Figure5_Regression.ipynb` loads.
mouse_recdays = list(np.load(os.path.join(META, 'combined_ABCDonly_days.npy')).astype(str))
print(f'{len(mouse_recdays)} recdays in combined_ABCDonly_days.npy')
print('first 3:', mouse_recdays[:3])

In [ ]:
# Build the LEC-shaped data_dic from the PFC directory layout.
# compute_norm=False: V4 works entirely in raw time and never touches Neurons_norm, so the
# ~4 GB of normalised arrays the v2 path needed are pure overhead here.
from glm_analysis_v2 import build_data_dic_from_pfc

data_dic = build_data_dic_from_pfc(DATA_FOLDER, mouse_recdays, compute_norm=False)
mouse_recdays = sorted(data_dic.keys())
print(f'\n{len(mouse_recdays)} recdays loaded')

In [ ]:
# Session selection, in three steps.
#
# 1. MIN_TRIALS. El-Gaby's `non_repeat_ses_maker` keeps any session with num_trials > 0; V4 used
#    >= 5. Seven PFC sessions in four recdays have 1-4 trials. The reproduction uses his rule.
# 2. Keep one session per unique task. This is exactly his `non_repeat_ses_maker`: both dedup by
#    exact array equality of the reward sequence.
# 3. Apply his one hand-exclusion. me11 session 3's task [7,4,3,8] shares 3 of 4 goals with
#    session 0's [7,4,3,5] -- his comment is "almost identical to session 0 (mistake)". Note that
#    his own cell 26 applies it inconsistently (7 folds fitted, 6 scored, positions 3-5
#    misaligned); we apply his intent cleanly, so me11's 46 neurons are an irreducible difference.
import importlib
import elasticnet_regression_v5 as v5
importlib.reload(v5)

MIN_TRIALS = 1          # his rule; V4 used 5

valid_sessions_dic = {}
for mouse_recday in mouse_recdays:
    valid_sessions, tasks = [], []
    for session in sorted(s for s in data_dic[mouse_recday] if s != 'valid_sessions'):
        sd = data_dic[mouse_recday][session]
        if sd['num_trials'] < MIN_TRIALS:
            print(f'{mouse_recday} session {session}: {sd["num_trials"]} trials < MIN_TRIALS, skipping')
            continue
        if not any(np.array_equal(sd['Task'], c) for c in tasks):
            tasks.append(sd['Task'])
            valid_sessions.append(session)
    valid_sessions_dic[mouse_recday] = valid_sessions

print('\nEl-Gaby hand-exclusions:')
valid_sessions_dic = v5.apply_excluded_sessions(valid_sessions_dic)
n_fold = {mr: len(v) for mr, v in valid_sessions_dic.items()}
print(f'\nfolds per recday: min {min(n_fold.values())}  median '
      f'{int(np.median(list(n_fold.values())))}  max {max(n_fold.values())}')
print('recdays with <2 folds (will be skipped):',
      [mr for mr, n in n_fold.items() if n < 2] or 'none')

## Run - past and future lags

Both directions, same config otherwise. `n_jobs` uses joblib's threading backend, so `data_dic` is shared rather than copied to workers.

In [ ]:
import importlib, os
from datetime import datetime
import elasticnet_regression_v5 as v5
importlib.reload(v5)

STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
N_JOBS = 6                      # 8 cores on this box; threading, so memory is shared
# 1252 units over 25 recdays, ~6 folds each -> ~3 h per direction at N_JOBS=6 for ElasticNet
# (the box is often loaded). Poisson (his executed path) is ~7x slower.
# 'none' = raw 25 ms spike counts, the reproduction. 'zscore_recday' divides each neuron's fit
# target by its sd over ALL used sessions of the recday (one scalar per neuron, so between-session
# rate differences within a recday survive), because a FIXED alpha=0.01 on raw counts is a
# firing-rate filter: it zeroes every coefficient for ~60% of neurons, and it does so as a
# function of rate. Such runs are written to `*_zscore_v5_*` directories and are NOT the
# reproduction. Long runs go through `../../code/slurm_v5/submit_v5_run.sh` (job 6 = PFC past,
# 7 = PFC future), not this notebook.
Y_SCALING = 'none'


def make_config(direction, **kw):
    # Everything not named here comes from the V5 defaults, each read off El-Gaby's deposited
    # code (see ELGABY_FIGURE5_RECONCILIATION.md):
    #   state_tuning_statistic='pref_phase_mean'   his Figure2 cell 46 (paper text says 'max')
    #   state_tuning_nan_policy='propagate'        his scipy.stats.zscore + ttest_1samp
    #   pref_phase_method='elgaby_peak'            his tuning_phase_boolean_max (cell 31)
    #   nonzero_lag_zero_lags=(0, 11)              "lag from anchor of 30 degrees or more"
    #   state_tuning_min_fraction=1/3              "state-tuned in more than one-third of the tasks"
    #   pref_phase_source='test'                   his cell 21 -- LEAKAGE, warned at run time
    #   poisson_link='linear'                      his readout; a Poisson run stores exp(Xb+b) too
    # Legacy (v4) behaviour is one call away:
    #   make_config(d, state_tuning_statistic='max', state_tuning_nan_policy='omit',
    #               pref_phase_method='raw_mean')
    # merge before the call: passing e.g. alpha_mode in **kw would otherwise collide with the
    # explicit keyword and raise TypeError
    params = dict(
        use_poisson=False, regularize=True,      # ElasticNet, alpha=0.01, positive
        lag_direction=direction,
        alpha_mode='fixed',                      # 'relative' + alpha_frac to rescale per neuron
        state_reduce='mean',                     # 'max' also stored either way
        y_scaling=Y_SCALING,                     # 'zscore_recday' -> a `*_zscore_v5_*` run
    )
    params.update(kw)
    return v5.RegressionConfigV5(**params)


results, pooled, diagnostics = {}, {}, {}
for direction in ('past', 'future'):
    cfg = make_config(direction)
    # Folder name leads with the estimator -- it is the setting that most changes the numbers.
    # Everything else, including MIN_TRIALS, is in run_config.json beside the arrays.
    out_dir = os.path.join('../data/figures', v5.run_dir_name(cfg, stamp=STAMP))
    print(f'\n{"="*70}\n{direction.upper()} lags -> {out_dir}\n{"="*70}')
    results[direction], pooled[direction], diagnostics[direction] = v5.run_and_summarise_all_mice_v5(
        data_dic, cfg,
        valid_sessions_dic=valid_sessions_dic,
        save_dir=out_dir, export_dir=out_dir,
        make_pdfs=True, n_jobs=N_JOBS, verbose=True,
        manifest_extra={'min_trials': MIN_TRIALS, 'y_scaling': Y_SCALING,
                        'notebook': 'PFC_elasticnet_regression_v5'})
configs = {d: make_config(d) for d in results}

### Per-recday diagnostics

Read this before the region table. `frac_allzero_fits` is the alpha dropout, `frac_pref_phase_flips` the fraction of neurons whose coordinate frame changes between folds, and `n_nonzero_lag_alt_top3` the mask size with the argsort tie-breaking guard flipped - a large gap there means the mask is partly sort-order artefact.

In [ ]:
for direction, tab in diagnostics.items():
    print(f'\n===== {direction} =====')
    print(tab.to_string(index=False))

### The state-tuning filter: his statistic vs the paper's text

`selected = nonzero_lag & state_tuned & r.notna()`. V4 implemented the paper's text ("peak
firing rate in each state and trial", `state_tuning_statistic='max'`), which is **confounded
by leg duration**: `raw_to_norm` averages more raw bins into each normalised bin when a leg is
longer, lowering its variance and so its max, so constant-rate cells acquire the shortest leg
as their "preferred state" (measured FPR on pure noise: 0.05 at equal legs, 0.97 at 2x, 1.00 at
3x; this dataset's median longest:shortest ratio is 1.82x).

**El-Gaby's code does not use the peak.** Figure2 cell 46 takes the mean over the neuron's
preferred-phase bins per (trial, state) and lets NaN propagate through `zscore` and the t-test.
That statistic is not a max-of-averaged-bins and control 9 of the synthetics measures its FPR
at 3x legs near nominal. So "the reference has the confound too" was wrong: it applies to the
paper's text, not to what produced the figures. V5's default is his statistic; `'max'` is
still computed as `state_tuned_mask_alt` so the swap stays visible below.

The NaN propagation matters on its own: a trial with all-zero preferred-phase means makes the
whole session "not tuned" for that neuron, which removes ~15% of PFC neuron-sessions and
preferentially low-rate units (his tuned units average 7.5 Hz vs 3.8 Hz untuned).

In [ ]:
# The tuning filter under his statistic vs the paper's 'max', the leg-duration ratio, and the
# skip bookkeeping. `n_selected_anyfold` / `n_tuned_finite_all_folds` are his semantics.
for direction, tab in diagnostics.items():
    print(f'\n===== {direction} =====')
    print(tab[['mouse_recday', 'n_neurons', 'n_folds', 'n_sessions_skipped', 'n_state_tuned',
               'n_state_tuned_alt_stat', 'n_tuned_finite_all_folds', 'n_selected',
               'n_selected_anyfold', 'state_duration_ratio',
               'frac_pref_state_is_shortest']].to_string(index=False))

# The same selection table under the OTHER statistic ('max' when the run used his). If a result
# only exists under 'max', it is a leg-geometry result.
tab = tables['past'].copy()
tab['selected'] = tab.nonzero_lag & tab.state_tuned_alt_stat & tab.mean_corr_nonzero.notna()
print("\n===== past lags, state tuning by the OTHER statistic (state_tuned_mask_alt) =====")
v5.region_summary(tab)

## Which animals do these neurons come from?

PFC has no anatomy, so `build_unit_table` falls back to identity columns (`recday`, `mouse`,
`order` = the `Neuron_raw` row index) and groups by animal. Every regression column is the same
one the LEC table carries.

In [ ]:
tables = {d: v5.build_unit_table(results[d], configs[d], data_dic=data_dic)
          for d in results}
print('has_anatomy:', {d: t.attrs['has_anatomy'] for d, t in tables.items()})
print('links:', {d: t.attrs['links'] for d, t in tables.items()})
# how often his preferred-phase rule and the v4 raw-mean rule agree, per neuron (mean over sessions)
print('pref-phase agreement (elgaby_peak vs raw_mean):',
      {d: round(float(t.pref_phase_agreement.mean()), 3) for d, t in tables.items()})
for direction, tab in tables.items():
    print(f'\n{"="*70}\n{direction.upper()} lags - selection by mouse\n{"="*70}')
    v5.region_summary(tab)

### The paper's three panels, under both semantics

El-Gaby reports the same correlation under three selections: all state-tuned, non-zero-lag
(30 deg, excluding lags {0,11}) and strict non-zero-lag (90 deg, excluding {0,1,2,9,10,11}).
The `elgaby` block is the **reproduction claim**: the pool is state-tuned neurons with a finite
r in every fold ("with non-zero beta coefficients"), a non-zero-lag neuron needs >= 1 passing
fold and is valued by the mean over passing folds. The `v4` block is the per-neuron
majority-vote mask with all folds averaged. Targets (ElasticNet, p<0.05): n = 489 / 329 / 224,
t = 9.3 / 3.9 / 2.53; at p<0.01 (ED 8b): 349 / 227 / 154. A Poisson run prints one block per
link and compares to ED 8d (489 / 346 / 229).

In [ ]:
# The paper's three panels, from the same fits, one block per (link, semantics).
panel_tables = {}
for direction, tab in tables.items():
    print(f'\n{"="*74}\n{direction.upper()} lags\n{"="*74}')
    panel_tables[direction] = v5.three_panel_summary(tab)

# The residual to watch is the 90-degree panel under 'elgaby' semantics (92 vs 224 on the v4 run).
past = panel_tables['past']
print('\n90-degree panel, past lags:')
print(past[past.panel.str.contains('90deg')].to_string(index=False))

### Past vs future

`pro_index = (r_future - r_past) / (|r_future| + |r_past|)`: positive means a unit is better
explained prospectively. The two designs are not degenerate - past lag *k* and future lag 12-*k*
point at the same task position one loop apart, and their measured column correlation is only
~0.02-0.34, because routes vary between trials.

In [ ]:
merged, direction_summary = v4.compare_directions(tables['past'], tables['future'])
merged.head()

### The firing-rate confound

At `alpha=0.01` a unit is only fittable if it fires fast enough - **57%** of PFC neurons fit
all-zero, and the survivors are the fast ones - so a difference in *selection rate* can be a
difference in *firing rate*. `region_summary` prints the within-quartile version above. To see
how much is the penalty rather than the biology, re-run one recday with a per-neuron relative
alpha.

In [ ]:
# Rate-matched re-run of a single recday (cheap): every neuron sits at the same point on
# its own regularization path instead of a shared absolute alpha. 57% of PFC neurons fit
# all-zero at the fixed alpha, so this is the check that says how much of the result is the
# penalty rather than the biology.
mr = list(results['past'].keys())[0]
cfg_rel = make_config('past', alpha_mode='relative', alpha_frac=0.1)
res_rel = v5.run_cross_validated_regression_v5(
    data_dic, mr, cfg_rel, valid_sessions=valid_sessions_dic[mr], verbose=True)
tab_rel = v5.build_unit_table({mr: res_rel}, cfg_rel, data_dic=data_dic)
print('\n--- relative alpha ---')
v5.region_summary(tab_rel)
print('\n--- fixed alpha, same recday ---')
v5.region_summary(tables['past'][tables['past'].recday == mr])

### The z-scored-target variant

`y_scaling='zscore_recday'` divides each neuron's fit target by its sd over the whole recday.
It is *not* part of the reproduction: it exists because the reference's fixed `alpha=0.01` on raw
25 ms counts is a firing-rate filter (the ElasticNet zeroing threshold scales with sd(y)), so the
selected population is partly a rate-selected population. Everything upstream of the fit -- the
state test, the preferred phases, the actual tuning curves -- is invariant to a positive
per-neuron affine transform and is bit-identical between the two runs (synthetic control 12), so
the comparison below isolates the effect of the penalty.

The cell reads the newest raw and `_zscore` run directories for one direction off disk (the runs
themselves are launched with `slurm_v5/submit_v5_run.sh`) and asks two questions: how many fits
stop being all-zero, and whether the newly admitted neurons are the low-spike ones.

In [ ]:
# Raw vs recday-z-scored target, from the stored runs (no re-fit).
import elasticnet_v5_compare as cmp5
importlib.reload(cmp5)
import numpy as np, pandas as pd, glob, os

DIRECTION = 'past'
FIG_ROOT = '../data/figures'


def _load_table(run_dir):
    """Rebuild a unit table from a run directory's npz exports."""
    res, cfg = {}, None
    for path in sorted(glob.glob(os.path.join(run_dir, f'*_{DIRECTION}_arrays.npz'))):
        z = v5.load_regression_outputs(path)
        rd = os.path.basename(path).replace(f'_{DIRECTION}_arrays.npz', '')
        res[rd] = z
        cfg = cfg or v5._config_from_results(z)
    return v5.build_unit_table(res, cfg, data_dic=data_dic, require_anatomy=False), cfg


pair = {}
for label, tag in (('raw counts', None), ('recday z-score', 'zscore')):
    try:
        run_dir = cmp5.resolve_latest(FIG_ROOT, DIRECTION, 'elasticnet', tag=tag)
    except FileNotFoundError as exc:
        print(f'{label}: {exc}')
        continue
    tab, cfg = _load_table(run_dir)
    pair[label] = (tab, cfg, run_dir)
    print(f'{label:16s} <- {os.path.basename(run_dir)}  ({len(tab)} neurons)')

for label, (tab, cfg, _) in pair.items():
    print(f'\n===== {label} =====')
    v5.three_panel_summary(tab)
    allzero = float((tab.n_nonzero_betas == 0).mean())
    print(f'  all-zero fits (mean over folds == 0): {allzero:.3f}')

# Did the extra neurons come from the low-spike end? Selection rate by spike-count quintile.
# `n_spikes_recday` was added with the y_scaling option, so a run older than that carries NaN
# there; the quintiles are taken from whichever table actually has it.
sizes = {lab: len(t) for lab, (t, _, _) in pair.items()}
ref = next((t for t, _, _ in pair.values() if np.isfinite(t.n_spikes_recday).any()), None)
if len(pair) == 2 and len(set(sizes.values())) > 1:
    print(f'\nrun sizes differ ({sizes}) -- not aligning neuron for neuron')
elif len(pair) == 2 and ref is None:
    print('\nneither run stores n_spikes_recday (both predate y_scaling); re-export to compare')
elif len(pair) == 2:
    rows = []
    q = pd.qcut(ref.n_spikes_recday, 5, labels=False, duplicates='drop')
    for label, (tab, _, _) in pair.items():
        for qi in sorted(pd.unique(q.dropna())):
            m = (q == qi).values
            rows.append({'target': label, 'spike_quintile': int(qi),
                          'median_spikes': float(np.nanmedian(tab.n_spikes_recday[m])),
                          'n': int(m.sum()),
                          'frac_allzero': float((tab.n_nonzero_betas[m] == 0).mean()),
                          'frac_state_tuned': float(tab.state_tuned[m].mean()),
                          'frac_nonzero_lag': float(tab.nonzero_lag[m].mean()),
                          'mean_r_nonzero': float(np.nanmean(tab.mean_corr_nonzero[m]))})
    admission = pd.DataFrame(rows).pivot(index='spike_quintile', columns='target')
    print('\n=== selection by recday spike-count quintile (the noise-admission caveat) ===')
    print(admission.to_string())

## Inspecting one neuron

`fold_betas` returns each fold's beta matrix collapsed in *that fold's* frame - the un-averaged view of what the `*_foldbetas.pdf` pages show.

In [ ]:
mr = list(results['past'].keys())[0]
res = results['past'][mr]
top = np.argsort(np.nan_to_num(res['mean_tuning_correlations_pref'], nan=-np.inf))[::-1][:5]
print('top neurons by preferred-phase tuning r:', top.tolist())
ni = int(top[0])
print(f'neuron {ni}: pref phase per fold = {res["pref_phases"][ni].tolist()}, '
      f'peak lag = {res["peak_lags"][ni]}, r = {res["mean_corrs"][ni]:.3f}, '
      f'reduced-beta r = {res["mean_corrs_nonzero"][ni]:.3f} '
      f'(passing folds only: {res["mean_corrs_nonzero_passing"][ni]:.3f}, '
      f'{int(res["n_passing_folds"][ni])} of {res["num_sessions"]}), '
      f'non-zero betas/fold = {res["n_nonzero_betas"][ni].tolist()}')
fb = v5.fold_betas(res, configs['past'], ni)          # (n_folds, 9 locations, 12 lags)
fig, axes = plt.subplots(1, len(fb) + 1, figsize=(2.2 * (len(fb) + 1), 2.4))
vmax = np.nanmax(np.abs(fb)) or 1.0
for fi, ax in enumerate(axes[:-1]):
    ax.imshow(fb[fi], aspect='auto', cmap='viridis', vmin=0, vmax=vmax)
    ax.set_title(f'fold {fi} (pref {res["pref_phases"][ni, fi]})', fontsize=7)
    ax.tick_params(labelsize=5)
B, modal, n_used, n_tot = v5.betas_in_common_frame(res, configs['past'], ni)
axes[-1].imshow(B, aspect='auto', cmap='viridis', vmin=0, vmax=vmax)
axes[-1].set_title(f'mean [{n_used}/{n_tot} folds, pref {modal}]', fontsize=7)
axes[-1].tick_params(labelsize=5)
fig.suptitle(f'{mr} neuron {ni} - past lags (location x lag)', fontsize=9)
fig.tight_layout()

# the same neuron as polar rate maps per held-out task (Fig 5g style), reduced-beta prediction solid
v5.plot_fold_polar_pages(res, configs['past'], f'../data/figures/{mr}_neuron{ni}_polar.pdf',
                         neuron_indices=[ni], prediction='nz')

## Reading the exports back

One `.npz` per recday per direction, plus eight PDFs:

| file | what it shows |
|---|---|
| `*_all.pdf` / `*_nonzerolag.pdf` | summary page per neuron: betas, 360-bin curves, n=4 readout |
| `*_all_foldbetas.pdf` / `*_nonzerolag_foldbetas.pdf` | beta matrix **per fold**, each in its own frame |
| `*_all_foldratemaps.pdf` / `*_nonzerolag_foldratemaps.pdf` | actual vs predicted **per fold**, linear axes |
| `*_all_polar.pdf` / `*_nonzerolag_polar.pdf` | actual vs predicted **per fold as polar rate maps** (Fig 5g style); on the `_nonzerolag` pages the solid predicted curve is the **reduced-beta** prediction, i.e. what that neuron's r is computed from |

For a Poisson run the other link's prediction is drawn too (dotted, Saffron) with both r values
in the titles, and the cross-mouse summary SVG has one row of panels per link.

The per-fold pages are the honest view: each fold holds out a *different task*, so the actual
tuning curve genuinely differs between columns, and the summary page's fold-averaged actual
blends them. A single fold's `r` comes from 4 points, whose null sampling SD is 1/sqrt(3) = 0.58:
read the shapes, not the individual numbers.

Nothing below needs `data_dic`. `run_config.json` records the config, `MIN_TRIALS`, the
sessions used and skipped per recday (with reasons), and the library versions.

In [ ]:
import glob, json
out_dir = os.path.join('../data/figures',
                       v5.run_dir_name(configs['past'], stamp=STAMP))
print('\n'.join(sorted(os.path.basename(p) for p in glob.glob(out_dir + '/*'))[:14]))
z = v5.load_regression_outputs(glob.glob(out_dir + '/*_arrays.npz')[0])
for k in sorted(z):
    print(f'  {k:40s} {getattr(z[k], "shape", type(z[k]).__name__)}')
man = json.load(open(os.path.join(out_dir, 'run_config.json')))
print('\nmanifest: min_trials =', man.get('min_trials'), '| sessions skipped in total =',
      man.get('sessions_skipped_total'))

## LEC vs PFC

[`elasticnet_v5_compare.py`](../../code/elasticnet_v5_compare.py) reads the exported `.npz`
files from both datasets - no `data_dic`, no re-fit. Needs the LEC v5 sweep to have been run too.

The unit of analysis is a **mouse_recday** (mean +/- SEM across recdays), and `min_neurons=2`
drops degenerate recdays - `me10_20122021_21122021` has exactly one neuron. `mean_r` is now the
reduced-beta r of the selected units (the all-betas r is kept as `mean_r_all_betas`).

Read the diagnostics table alongside the result. The selection criterion carries a large
baseline on both datasets (the non-zero-lag test fires on ~30% of pure Poisson noise at the
30-degree set), and `compare_datasets` warns if the two runs were produced with different
settings -- a reproduction-config PFC run (`MIN_TRIALS=1`, test-session preferred phase) is not
directly comparable to a science-config LEC run (`MIN_TRIALS=5`, training-session preferred
phase); compare like with like.

In [ ]:
import sys
sys.path.insert(0, '../../code')
import elasticnet_v5_compare as cmp
importlib.reload(cmp)

# ESTIMATOR pins which run to compare -- folders are named {estimator}_v5_{direction}_{stamp},
# and compare_datasets also reads each run_config.json and warns if the settings differ.
ESTIMATOR = v5.estimator_name(configs['past'])
dirs = {ds: {d: cmp.resolve_latest(root, d, ESTIMATOR) for d in ('past', 'future')}
        for ds, root in cmp.DEFAULT_ROOTS.items()}
for ds, by_dir in dirs.items():
    for d, path in by_dir.items():
        print(f'{ds} {d}: {path}')
stats, table, fig = cmp.compare_datasets(
    dirs, min_neurons=2,
    out_path=f'../data/figures/{ESTIMATOR}_v5_lec_vs_pfc')